# RAG benchmark · directly from DeepEval Goldens

Load `rag-goldens/deepeval-goldens.json` with **DeepEval's own JSON importer**,
run each question with the tiny observed `cli-agent` SDK target, and score
with a **separately selected** DeepEval judge. No generator scripts or
`sys.path` manipulation are required.

The agent only sees the question; reference answers and source metadata
belong to the evaluator. Selected Knowledge passages are taken only from
the actual SDK trace, never from golden evidence.

## 1. Configuration

Start with three cases. Set `MAX_CASES=None` after verifying the setup.
Use an appropriate judge provider with permission to receive reference
answers and optionally retrieved passages.

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import json
import os
import tomllib
import tempfile

WORKSPACE = Path.cwd().resolve()
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "deepeval-goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

CONFIG_FILE = None             # cli-agent config.toml, None = installed default
ADMIN_CONFIG_FILE = None       # None = cli-agent admin defaults
APPROVE_TOOLS = ()             # exact names; unlisted tools are not auto-approved
MAX_CASES = 3                  # None = all goldens
SDK_TIMEOUT_SECONDS = 600
FRESH_SDK = False              # force rerun after external index/corpus changes
CACHE_EPOCH = ""               # change this for other untracked target changes

# Completely independent DeepEval judge
JUDGE_PROVIDER = "openrouter"  # openrouter | openai | ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False

## 2. Load DeepEval Goldens and build the target + judge

`EvaluationDataset.add_goldens_from_json_file()` reads the export directly.
The SDK's `AgentBuilder.build_deepeval_target()` owns the target integration;
the judge never inherits the target model.

In [ ]:
from cli_agent.sdk import AgentBuilder
from cli_agent.settings.config import ensure_default_config_file
from deepeval.dataset import EvaluationDataset
from deepeval.models import OpenAIModel, OpenRouterModel, OllamaModel

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(str(GOLDENS_FILE))
if MAX_CASES is not None:
    dataset = EvaluationDataset(goldens=dataset.goldens[:MAX_CASES])
if not dataset.goldens:
    raise ValueError("No exported DeepEval Goldens found")

project_config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
admin_config = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else None
builder = AgentBuilder().with_admin_config(path=admin_config)
builder.with_project_config(
    path=project_config, workspace=WORKSPACE,
    workspace_access="none", response_format="text",
)
if APPROVE_TOOLS:
    allowed = frozenset(APPROVE_TOOLS)
    async def approve(name, arguments):
        return name in allowed
    builder.with_approval_callback(approve)
target = builder.build_deepeval_target(include_knowledge_content=True)

if JUDGE_PROVIDER == "openrouter":
    judge = OpenRouterModel(model=JUDGE_MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
elif JUDGE_PROVIDER == "openai":
    judge = OpenAIModel(model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"],
                        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}))
elif JUDGE_PROVIDER == "ollama":
    judge = OllamaModel(model=JUDGE_MODEL,
                        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}))
else:
    raise ValueError(f"Unsupported JUDGE_PROVIDER: {JUDGE_PROVIDER}")
if JUDGE_PROVIDER == "openrouter" and JUDGE_BASE_URL:
    raise ValueError("OpenRouter uses its fixed endpoint; remove JUDGE_BASE_URL")
print(f"{len(dataset.goldens)} Goldens | judge: {JUDGE_PROVIDER}/{JUDGE_MODEL}")

## 3. Execute observed target questions inside DeepEval

DeepEval's `evals_iterator()` coordinates each observed target call.
A tiny local **no-cost** response check fulfils the iterator's metric
requirement; the actual judge metrics run through DeepEval's cache-aware
`evaluate()` below.

Successful target responses are stored separately and replayed through
the observed SDK adapter, allowing changes of the judge without rerunning
the target. The cache fingerprint includes `project.toml` **and the contents
of its referenced system prompt file**. Set `FRESH_SDK=True` when changing
external knowledge indexes, MCP tools or other untracked runtime state.

In [ ]:
from deepeval.metrics import BaseMetric
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

class AnswerCaptured(BaseMetric):
    threshold = 1.0
    async_mode = False
    def measure(self, test_case, *args, **kwargs):
        self.score = float(bool((test_case.actual_output or "").strip()))
        self.success = self.is_successful()
        return self.score
    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case)
    @property
    def __name__(self):
        return "Target answer captured"

def hash_config():
    from cli_agent.settings.admin_config import default_admin_config_file
    files = [Path(project_config)]
    policy_file = Path(admin_config) if admin_config else default_admin_config_file()
    if policy_file.is_file():
        files.append(policy_file)
    base = tomllib.loads(Path(project_config).read_text(encoding="utf-8"))
    prompt_name = base.get("agent", {}).get("system_prompt")
    override = WORKSPACE / ".cli-agent" / "project.toml"
    if override.is_file():
        files.append(override)
        project = tomllib.loads(override.read_text(encoding="utf-8"))
        prompt_name = project.get("agent", {}).get("system_prompt", prompt_name)
    if prompt_name:
        files.append(WORKSPACE / ".cli-agent" / prompt_name)
    digest = hashlib.sha256()
    for file in files:
        file = file.expanduser().resolve(strict=True)
        digest.update(str(file).encode("utf-8"))
        digest.update(file.read_bytes())
    digest.update(str(sorted(APPROVE_TOOLS)).encode("utf-8"))
    digest.update(CACHE_EPOCH.encode("utf-8"))
    return digest.hexdigest()

target_config_hash = hash_config()
cache_dir = OUTPUT_DIR / "responses"
cache_dir.mkdir(parents=True, exist_ok=True)
options = {
    "cache_config": CacheConfig(use_cache=not FRESH_EVALUATION, write_cache=True),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
        inspect_after_run=False,
    ),
}
responses = {}
for golden in dataset.evals_iterator(
    metrics=[AnswerCaptured()], identifier="rag-target-execution",
    async_config=AsyncConfig(run_async=False), **options,
):
    metadata = golden.additional_metadata or {}
    golden_id = metadata["golden_id"]
    key = hashlib.sha256(json.dumps(
        [target_config_hash, golden.input, golden.expected_output, golden_id],
        ensure_ascii=False,
    ).encode("utf-8")).hexdigest()
    checkpoint = cache_dir / f"{key}.json"
    cached = None
    if (checkpoint.is_file() and not checkpoint.is_symlink()
        and checkpoint.stat().st_size <= 16_000_000 and not FRESH_SDK):
        record = json.loads(checkpoint.read_text(encoding="utf-8"))
        if (isinstance(record, dict)
            and isinstance(record.get("actual_output"), str)
            and record["actual_output"].strip()
            and (record.get("knowledge_trace") is None
                 or isinstance(record["knowledge_trace"], dict))):
            cached = record
    async with asyncio.timeout(SDK_TIMEOUT_SECONDS):
        response = await target.ask(
            golden.input, expected_output=golden.expected_output,
            cached_response=cached,
        )
    if cached is None:
        # Private, atomic output: answers may include retrieved Knowledge text.
        temp_name = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", encoding="utf-8", dir=cache_dir,
                prefix=".target-", suffix=".tmp", delete=False,
            ) as stream:
                temp_name = Path(stream.name)
                os.chmod(temp_name, 0o600)
                json.dump(response, stream, ensure_ascii=False)
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temp_name, checkpoint)
        finally:
            if temp_name is not None:
                temp_name.unlink(missing_ok=True)
    responses[golden_id] = response

print(f"Observed {len(responses)} target responses")

## 4. Native DeepEval judge evaluation

The **answer quality** evaluation scores every case against its expected
answer. Retrieval metrics are only applied where the target's SDK actually
returned complete selected Knowledge passages; the export does not supply
ideal passages as a replacement.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval, AnswerRelevancyMetric, FaithfulnessMetric,
    ContextualRecallMetric, ContextualPrecisionMetric,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams

answer_cases = []
retrieval_cases = []
for golden in dataset.goldens:
    response = responses[golden.additional_metadata["golden_id"]]
    base = dict(
        input=golden.input, expected_output=golden.expected_output,
        actual_output=response["actual_output"],
    )
    answer_cases.append(LLMTestCase(**base))
    trace = response.get("knowledge_trace") or {}
    selected = trace.get("selected") or []
    if (trace.get("content_included") is True
        and trace.get("truncated") is not True
        and selected and all(
            isinstance(hit, dict) and isinstance(hit.get("content"), str)
            and hit["content"].strip() for hit in selected
        )):
        retrieval_cases.append(LLMTestCase(
            **base, retrieval_context=[hit["content"] for hit in selected]
        ))

correctness = GEval(
    name="Answer correctness",
    criteria=("Check that the answer is factually correct and complete "
              "against the reference; accept valid paraphrases, "
              "penalize missing caveats and invented facts."),
    evaluation_params=[
        SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=judge, threshold=THRESHOLD,
)
answer_results = evaluate(
    test_cases=answer_cases,
    metrics=[correctness, AnswerRelevancyMetric(model=judge, threshold=THRESHOLD)],
    identifier="rag-answer-quality",
    async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
    **options,
)

if retrieval_cases:
    retrieval_results = evaluate(
        test_cases=retrieval_cases,
        metrics=[
            FaithfulnessMetric(model=judge, threshold=THRESHOLD),
            ContextualRecallMetric(model=judge, threshold=THRESHOLD),
            ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
        ],
        identifier="rag-retrieval-quality",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **options,
    )
else:
    print("No verified selected Knowledge passages: retrieval metrics skipped.")
print("DeepEval reports:", OUTPUT_DIR / "deepeval")

## Notes

- Native input: `rag-goldens/deepeval-goldens.json`, generated by the
  exporter. No generator `scripts` imports, `sys.path`, or custom dataset
  conversion are needed.
- `goldens.json` remains the rich, auditable export with source hashes,
  quotes and review evidence.
- Judge calls are independent of the SDK model and can use another provider.
- Target answer checkpoints are local under `evaluation/responses/`.
  Change `FRESH_SDK` or `CACHE_EPOCH` after changes to the live RAG index.
- Contextual metrics use only **actual** retrieved passages; source
  metadata in the Golden is not retrieval context. Saved traces and judge
  requests may contain confidential data.